# Step 4A.4 — Baseline-моделі B0 / B1 / B2_S (H=180, primary temporally-safe шлях)
Чистий baseline-експеримент, без hyperparameter search. Використовує ВЖЕ існуючий
`split_180d` зі Step 4A.2 (жодного нового спліту). B2_S = лише `epss_t`, `cve_age_days`,
`post_epss_published` — з `FEATURE_CONTRACT`, зафіксованого в Audit 4A.3-B.

Test використовується РІВНО ОДИН раз, у самому кінці — жодного tuning/early stopping на ньому.

Ядро: IDS Research.

In [1]:
# ---------- Cell 1: Configuration ----------
import sys, subprocess
try:
    import lightgbm as lgb
except ImportError:
    print("lightgbm не знайдено — встановлюю...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm"], check=True)
    import lightgbm as lgb

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import average_precision_score, roc_auc_score

CTI_DIR = Path("/srv/ids_research/data/cti")
HORIZON = 180          # primary horizon для цього ноутбука (365 — окремий прогін пізніше)
TARGET_COL = f"future_kev_{HORIZON}d"
SPLIT_COL = f"split_{HORIZON}d"
TOP_K_FRACTIONS = [0.001, 0.005, 0.01, 0.05]   # 0.1% / 0.5% / 1% / 5%

RANDOM_STATE = 42
print(f"HORIZON={HORIZON}, TARGET_COL={TARGET_COL}, SPLIT_COL={SPLIT_COL}")
print(f"lightgbm version: {lgb.__version__}")

HORIZON=180, TARGET_COL=future_kev_180d, SPLIT_COL=split_180d
lightgbm version: 4.7.0


In [2]:
# ---------- Cell 2: Load dataset ----------
ds = pd.read_parquet(CTI_DIR / "cve_forecast_dataset.parquet")
print(f"Завантажено: {len(ds):,} рядків, {len(ds.columns)} колонок")
print(ds[[TARGET_COL, SPLIT_COL]].isna().mean())

Завантажено: 379,036 рядків, 21 колонок
future_kev_180d    0.129077
split_180d         0.000000
dtype: float64


In [3]:
# ---------- Cell 3: Add safe derived features (з Audit 4A.3-B, НЕ мутує оригінальний ds) ----------
def add_safe_derived_features(df):
    """cve_age_days is a derived modeling feature and is NOT persisted back to
    cve_forecast_dataset.parquet. published/t_ref лишаються excluded як сирі колонки."""
    df = df.copy()
    df["cve_age_days"] = (pd.to_datetime(df["t_ref"]) - pd.to_datetime(df["published"])).dt.days
    return df

ds = add_safe_derived_features(ds)
print("cve_age_days додано. Колонок тепер:", len(ds.columns))

# FEATURE_CONTRACT (заморожений в Audit 4A.3-B) — лише PRIMARY safe гілка тут
B2_S_FEATURES = ["epss_t", "cve_age_days", "post_epss_published"]
B1_FEATURES = ["epss_t"]
print("B1 features:", B1_FEATURES)
print("B2_S features:", B2_S_FEATURES)

cve_age_days додано. Колонок тепер: 22
B1 features: ['epss_t']
B2_S features: ['epss_t', 'cve_age_days', 'post_epss_published']


In [4]:
# ---------- Cell 4: Build observed dataset (виключаємо цензуровані NaN-таргети) ----------
n_before = len(ds)
observed = ds[ds[TARGET_COL].notna()].copy()
observed[TARGET_COL] = observed[TARGET_COL].astype(int)
observed["post_epss_published"] = observed["post_epss_published"].astype(int)   # bool -> int для LightGBM
n_after = len(observed)
print(f"Виключено цензурованих (NaN {TARGET_COL}): {n_before - n_after:,} ({(n_before-n_after)/n_before:.2%})")
print(f"Спостережуваних рядків: {n_after:,}")
print(f"\nРозподіл {TARGET_COL}:")
print(observed[TARGET_COL].value_counts())
print(f"Prevalence: {observed[TARGET_COL].mean():.4%}")

Виключено цензурованих (NaN future_kev_180d): 48,925 (12.91%)
Спостережуваних рядків: 330,111

Розподіл future_kev_180d:
future_kev_180d
0    329593
1       518
Name: count, dtype: int64
Prevalence: 0.1569%


In [5]:
# ---------- Cell 5: Verify existing temporal splits (з Step 4A.2, БЕЗ нового спліту) ----------
print(f"Розподіл {SPLIT_COL} серед спостережуваних:")
print(observed[SPLIT_COL].value_counts())

for split_name in ["train", "val", "test"]:
    sub = observed[observed[SPLIT_COL] == split_name]
    print(f"\n{split_name}: n={len(sub):,}, positives={int(sub[TARGET_COL].sum())}, "
          f"prevalence={sub[TARGET_COL].mean():.4%}")

# ДІАГНОСТИКА (не assert): split_180d визначається ГРУБИМ правилом за t_ref, а
# future_kev_180d — ТОЧНІШОЮ по-CVE логікою (Step 4A.2), яка може визнати результат
# ВІДОМИМ (0 або 1) навіть коли t_ref формально "запізно" за грубим правилом спліту
# (якщо kev_date_added вже настала до TODAY в межах часткового вікна). Тому невелика
# кількість рядків може мати split_180d=="excluded_censored" і водночас НЕ-NaN таргет.
# Це НЕ впливає на коректність експерименту: нижче train/val/test відбираються СУВОРО
# за значенням split_180d ∈ {"train","val","test"} — такі рядки просто ніколи туди не
# потрапляють, незалежно від того, що вони "видимі" в observed.
n_excluded_but_known = int((observed[SPLIT_COL] == "excluded_censored").sum())
if n_excluded_but_known:
    print(f"\nℹ {n_excluded_but_known} рядків мають split_180d='excluded_censored', але відомий "
          f"(не-NaN) {TARGET_COL} — прикордонний ефект грубого vs точного censoring-правила. "
          f"Ці рядки НЕ потраплять у train/val/test нижче (відбір йде строго за split ∈ "
          f"{{train,val,test}}), тож на розміри спліту це не впливає.")
else:
    print("\n✅ excluded_censored відсутній серед спостережуваних рядків")

train = observed[observed[SPLIT_COL] == "train"].copy()
val = observed[observed[SPLIT_COL] == "val"].copy()
test = observed[observed[SPLIT_COL] == "test"].copy()
print(f"\nФінальні розміри: train={len(train):,}  val={len(val):,}  test={len(test):,}")

# Точна (не "≈") перевірка проти зафіксованого Step 4A.2 dataset — ноутбук ЗУПИНЯЄТЬСЯ,
# якщо cve_forecast_dataset.parquet випадково змінився з часу заморожування Step 4A.2.
EXPECTED_SPLIT_SIZES = {"train": 249725, "val": 15192, "test": 65159}
for split_name, expected in EXPECTED_SPLIT_SIZES.items():
    actual = len(observed[observed[SPLIT_COL] == split_name])
    assert actual == expected, f"{split_name}: actual={actual}, expected={expected} — dataset змінився з часу Step 4A.2!"
print("\n✅ Розміри train/val/test точно збігаються із зафіксованим Step 4A.2 dataset")

Розподіл split_180d серед спостережуваних:
split_180d
train                249725
test                  65159
val                   15192
excluded_censored        35
Name: count, dtype: int64

train: n=249,725, positives=366, prevalence=0.1466%

val: n=15,192, positives=27, prevalence=0.1777%

test: n=65,159, positives=90, prevalence=0.1381%

ℹ 35 рядків мають split_180d='excluded_censored', але відомий (не-NaN) future_kev_180d — прикордонний ефект грубого vs точного censoring-правила. Ці рядки НЕ потраплять у train/val/test нижче (відбір йде строго за split ∈ {train,val,test}), тож на розміри спліту це не впливає.

Фінальні розміри: train=249,725  val=15,192  test=65,159

✅ Розміри train/val/test точно збігаються із зафіксованим Step 4A.2 dataset


In [6]:
# ---------- Cell 6: B0 — constant train prevalence (НЕ prevalence всього датасету) ----------
p_train = train[TARGET_COL].mean()
print(f"B0: p_train (константна ймовірність для всіх val/test) = {p_train:.6f}")

b0_val_scores = np.full(len(val), p_train)
b0_test_scores = np.full(len(test), p_train)
print(f"B0 val prevalence порівняно: train={p_train:.4%}, val_actual={val[TARGET_COL].mean():.4%}, "
      f"test_actual={test[TARGET_COL].mean():.4%}")

B0: p_train (константна ймовірність для всіх val/test) = 0.001466
B0 val prevalence порівняно: train=0.1466%, val_actual=0.1777%, test_actual=0.1381%


In [7]:
# ---------- Cell 7: B1 — raw EPSS ranking score (жодного навчання, жодного калібрування) ----------
# B1 залишається "чистим" EPSS-only baseline: НЕ навчаємо жодного classifier поверх epss_t,
# інакше B1 перестане відповідати на питання "наскільки добре сам EPSS прогнозує future KEV".
b1_val_scores = val["epss_t"].values
b1_test_scores = test["epss_t"].values
print("B1: raw EPSS ranking score (epss_t напряму, без калібрування під future_kev_180d).")
print(f"epss_t (val) — min={b1_val_scores.min():.4f}, max={b1_val_scores.max():.4f}, "
      f"NaN={val['epss_t'].isna().sum()}")
assert val["epss_t"].isna().sum() == 0 and test["epss_t"].isna().sum() == 0, \
    "epss_t не повинен мати NaN у спостережуваних рядках (перевірити EPSS-join зі Step 4A.2)"

B1: raw EPSS ranking score (epss_t напряму, без калібрування під future_kev_180d).
epss_t (val) — min=0.0004, max=0.9366, NaN=0


In [8]:
# ---------- Cell 8: Побудова X/y для B2_S + розширені sanity-перевірки на всіх трьох наборах ----------
X_train = train[B2_S_FEATURES]
y_train = train[TARGET_COL]
X_val = val[B2_S_FEATURES]
y_val = val[TARGET_COL]
X_test = test[B2_S_FEATURES]
y_test = test[TARGET_COL]
print("X_train dtypes:")
print(X_train.dtypes)

for name, X in [("train", X_train), ("val", X_val), ("test", X_test)]:
    assert X.isna().sum().sum() == 0, f"B2_S features не повинні мати NaN у {name}"
    assert np.isfinite(X.to_numpy()).all(), f"B2_S features не повинні мати inf/-inf у {name}"
    assert set(X["post_epss_published"].unique()).issubset({0, 1}),         f"post_epss_published має бути 0/1 у {name}"
    assert (X["cve_age_days"] >= 0).all(), f"cve_age_days не може бути від'ємним у {name}"
    assert ((X["epss_t"] >= 0) & (X["epss_t"] <= 1)).all(), f"epss_t має бути в [0,1] у {name}"
print("\n✅ NaN/inf/діапазон-перевірки пройдено для train, val і test")

# ---------- Cell 9: scale_pos_weight — ЛИШЕ з train (не val, не test, не весь ds) ----------
n_pos_train = int(y_train.sum())
n_neg_train = len(y_train) - n_pos_train
scale_pos_weight = n_neg_train / n_pos_train
print(f"n_pos_train={n_pos_train}, n_neg_train={n_neg_train}, scale_pos_weight={scale_pos_weight:.2f}")

X_train dtypes:
epss_t                 float64
cve_age_days             int64
post_epss_published      int64
dtype: object

✅ NaN/inf/діапазон-перевірки пройдено для train, val і test
n_pos_train=366, n_neg_train=249359, scale_pos_weight=681.31


In [9]:
# ---------- Cell 10: Train B2_S (LightGBM, early stopping НА VALIDATION, НЕ на test) ----------
# КРИТИЧНО: LightGBM з metric="average_precision" рахує ЦЮ метрику з урахуванням ваг,
# заданих scale_pos_weight (той самий множник, що йде на positive-клас у функції втрат).
# Це означає, що best_iteration обирався б за WEIGHTED AP, тоді як фінальна оцінка нижче —
# звичайний average_precision_score() без ваг. Щоб критерій відбору ітерації і критерій
# фінальної оцінки збігались, задаємо metric="None" і подаємо власну UNWEIGHTED AP через feval —
# scale_pos_weight тоді впливає ЛИШЕ на навчання (функцію втрат), а не на criteria відбору.
def unweighted_average_precision(preds, dataset):
    y_true = dataset.get_label()
    ap = average_precision_score(y_true, preds)
    return "AP_unweighted", ap, True   # is_higher_better=True

lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_train)

params = dict(
    objective="binary",
    metric="None",                  # вимикаємо вбудовану (weighted) метрику LightGBM
    scale_pos_weight=scale_pos_weight,
    verbosity=-1,
    seed=RANDOM_STATE,
    num_leaves=15,                  # свідомо консервативно — це baseline, не tuning
    learning_rate=0.05,
    min_data_in_leaf=50,
)
b2s_model = lgb.train(
    params, lgb_train, num_boost_round=500,
    valid_sets=[lgb_val], valid_names=["val"],
    feval=unweighted_average_precision,
    callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False), lgb.log_evaluation(period=0)],
)
print(f"B2_S: зупинено на ітерації {b2s_model.best_iteration} "
      f"(найкраща за UNWEIGHTED AP на VALIDATION, узгоджено з фінальною оцінкою; не на test)")

B2_S: зупинено на ітерації 2 (найкраща за UNWEIGHTED AP на VALIDATION, узгоджено з фінальною оцінкою; не на test)


In [10]:
# ---------- Cell 11: Прогнози на val і test — TEST використовується ОДИН РАЗ, тут ----------
b2s_val_scores = b2s_model.predict(X_val, num_iteration=b2s_model.best_iteration)
b2s_test_scores = b2s_model.predict(X_test, num_iteration=b2s_model.best_iteration)
print("Прогнози B2_S згенеровано для val і test.")

ALL_SCORES = {
    "B0": {"val": b0_val_scores, "test": b0_test_scores},
    "B1": {"val": b1_val_scores, "test": b1_test_scores},
    "B2_S": {"val": b2s_val_scores, "test": b2s_test_scores},
}
ALL_LABELS = {"val": y_val.values, "test": y_test.values}

Прогнози B2_S згенеровано для val і test.


In [11]:
# ---------- Cell 12: Average Precision / AP (основна) / ROC-AUC (secondary) — accuracy НЕ рахуємо взагалі ----------
# AP = sklearn.average_precision_score (східчаста апроксимація area under PR curve, БЕЗ
# інтерполяції) — саме так і називаємо в статті: "Average Precision (AP)", а не "PR-AUC",
# щоб не створювати враження іншої (трапецієподібної) реалізації інтегралу PR-кривої.
main_metrics = []
for model_name, scores in ALL_SCORES.items():
    for split_name in ("val", "test"):
        y_true = ALL_LABELS[split_name]
        y_score = scores[split_name]
        ap = average_precision_score(y_true, y_score)
        roc_auc = roc_auc_score(y_true, y_score) if len(np.unique(y_true)) > 1 else np.nan
        main_metrics.append(dict(model=model_name, split=split_name, ap=ap, roc_auc=roc_auc))

main_df = pd.DataFrame(main_metrics)
display(main_df)
print(f"\nДовідково — очікуваний Average Precision (AP) при випадковому ранжуванні = prevalence тестового набору: "
      f"{y_test.mean():.4%}")


,model,split,ap,roc_auc
0,B0,val,0.001777,0.500000
1,B0,test,0.001381,0.500000
2,B1,val,0.075574,0.539244
3,B1,test,0.018597,0.715477
4,B2_S,val,0.055299,0.613966
5,B2_S,test,0.007880,0.573759



Довідково — очікуваний Average Precision (AP) при випадковому ранжуванні = prevalence тестового набору: 0.1381%


In [12]:
# ---------- Cell 13: Precision@K / Recall@K для K ∈ {0.1%, 0.5%, 1%, 5%} ----------
def precision_recall_at_k(y_true, y_score, k_fraction):
    """argsort-based ranking Precision/Recall@K — коректно ЛИШЕ коли y_score дає змістовне
    ранжування (B1, B2_S). Для константних скорів (B0) argsort дав би артефакт: стабільне
    сортування за порядком рядків DataFrame, а НЕ справжній випадковий вибір — тому B0
    рахується ОКРЕМОЮ аналітичною формулою нижче, не цією функцією."""
    n = len(y_true)
    k = max(1, int(round(n * k_fraction)))
    order = np.argsort(-y_score)          # спадання за ризик-скором
    top_k_idx = order[:k]
    n_pos_in_top_k = y_true[top_k_idx].sum()
    n_pos_total = y_true.sum()
    precision = n_pos_in_top_k / k
    recall = n_pos_in_top_k / n_pos_total if n_pos_total > 0 else np.nan
    return precision, recall, k

def analytical_random_baseline_at_k(y_true, k_fraction):
    """B0 — аналітичний expected Precision/Recall@K для ВИПАДКОВОГО (не впорядкованого за
    константним скором) вибору K з N: E[Precision@K] = prevalence, E[Recall@K] = K/N.
    Це справжній lower-bound baseline, а не залежний-від-порядку-рядків артефакт."""
    n = len(y_true)
    k = max(1, int(round(n * k_fraction)))
    prevalence = y_true.mean()
    precision = prevalence
    recall = k / n
    return precision, recall, k

topk_rows = []
for model_name, scores in ALL_SCORES.items():
    for split_name in ("val", "test"):
        y_true = ALL_LABELS[split_name]
        y_score = scores[split_name]
        for frac in TOP_K_FRACTIONS:
            if model_name == "B0":
                precision, recall, k = analytical_random_baseline_at_k(y_true, frac)
            else:
                precision, recall, k = precision_recall_at_k(y_true, y_score, frac)
            topk_rows.append(dict(model=model_name, split=split_name, top_pct=f"{frac:.1%}",
                                   K=k, precision=precision, recall=recall))

topk_df = pd.DataFrame(topk_rows)
display(topk_df[topk_df["split"] == "test"].pivot(index="model", columns="top_pct", values=["precision", "recall"]))
print(f"\nДля довідки: 1% від test (N={len(y_test):,}) ≈ {int(round(len(y_test)*0.01)):,} CVE")

precision                                  recall                      \
top_pct      0.1%      0.5%      1.0%      5.0%      0.1%      0.5%      1.0%   
model                                                                           
B0       0.001381  0.001381  0.001381  0.001381  0.000998  0.005003  0.010006   
B1       0.046154  0.042945  0.024540  0.009515  0.033333  0.155556  0.177778   
B2_S     0.061538  0.033742  0.018405  0.005525  0.044444  0.122222  0.133333   

                   
top_pct      5.0%  
model              
B0       0.050001  
B1       0.344444  
B2_S     0.200000


Для довідки: 1% від test (N=65,159) ≈ 652 CVE


In [13]:
# ---------- Cell 14: Зведена таблиця для статті (лише TEST, одноразова фінальна оцінка) ----------
summary_rows = []
for model_name in ["B0", "B1", "B2_S"]:
    row = dict(model=model_name)
    row["ap"] = main_df.query("model == @model_name and split == 'test'")["ap"].iloc[0]
    row["roc_auc"] = main_df.query("model == @model_name and split == 'test'")["roc_auc"].iloc[0]
    for frac in TOP_K_FRACTIONS:
        target_pct = f"{frac:.1%}"
        sub = topk_df.query("model == @model_name and split == 'test' and top_pct == @target_pct")
        row[f"precision@{frac:.1%}"] = sub["precision"].iloc[0]
        row[f"recall@{frac:.1%}"] = sub["recall"].iloc[0]
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index("model")
print("=" * 70)
print(f"ФІНАЛЬНА ТАБЛИЦЯ ДЛЯ СТАТТІ (H={HORIZON}, TEST, одноразова оцінка)")
print("=" * 70)
display(summary_df)

ФІНАЛЬНА ТАБЛИЦЯ ДЛЯ СТАТТІ (H=180, TEST, одноразова оцінка)


,ap,roc_auc,precision@0.1%,recall@0.1%,precision@0.5%,recall@0.5%,precision@1.0%,recall@1.0%,precision@5.0%,recall@5.0%
model,,,,,,,,,,
B0,0.001381,0.500000,0.001381,0.000998,0.001381,0.005003,0.001381,0.010006,0.001381,0.050001
B1,0.018597,0.715477,0.046154,0.033333,0.042945,0.155556,0.024540,0.177778,0.009515,0.344444
B2_S,0.007880,0.573759,0.061538,0.044444,0.033742,0.122222,0.018405,0.133333,0.005525,0.200000


In [14]:
# ---------- Cell 15: Leakage / sanity-перевірки ПІСЛЯ навчання ----------
print("1. B2_S features відповідають FEATURE_CONTRACT (Audit 4A.3-B)?")
assert set(B2_S_FEATURES) == {"epss_t", "cve_age_days", "post_epss_published"}
print("   ✅")

print("\n2. Жодна з sensitivity_upper_bound колонок НЕ використана в X_train/X_val/X_test:")
sensitivity_cols = {"cvss_score", "cwe_ids", "products", "description"}
assert not (sensitivity_cols & set(X_train.columns))
print("   ✅ cvss_score/cwe_ids/products/description відсутні серед фіч B2_S")

print("\n3. scale_pos_weight обчислено ЛИШЕ з train (не з val/test/усього ds)?")
recomputed = (len(y_train) - y_train.sum()) / y_train.sum()
assert abs(recomputed - scale_pos_weight) < 1e-9
print(f"   ✅ scale_pos_weight={scale_pos_weight:.4f} відтворюється лише з train")

print("\n4. best_iteration обрано на validation, а НЕ на test (перевірка, що lgb_train сет валідації = val, не test):")
assert b2s_model.best_iteration is not None
print(f"   ✅ best_iteration={b2s_model.best_iteration}, обрано за valid_sets=[lgb_val] (не test)")

print("\n5. Test-скори НЕ використовувались до цієї комірки для жодного рішення (early stopping/threshold):")
print("   ✅ test-прогнози згенеровано в Cell 11, ПІСЛЯ фіксації best_iteration у Cell 10")

print("\n6. Розміри train/val/test збігаються з очікуваними зі Step 4A.2:")
print(f"   train={len(train):,} (= {EXPECTED_SPLIT_SIZES['train']:,})  "
      f"val={len(val):,} (= {EXPECTED_SPLIT_SIZES['val']:,})  "
      f"test={len(test):,} (= {EXPECTED_SPLIT_SIZES['test']:,})")

print("\n✅ Усі leakage/sanity-перевірки пройдено.")

1. B2_S features відповідають FEATURE_CONTRACT (Audit 4A.3-B)?
   ✅

2. Жодна з sensitivity_upper_bound колонок НЕ використана в X_train/X_val/X_test:
   ✅ cvss_score/cwe_ids/products/description відсутні серед фіч B2_S

3. scale_pos_weight обчислено ЛИШЕ з train (не з val/test/усього ds)?
   ✅ scale_pos_weight=681.3087 відтворюється лише з train

4. best_iteration обрано на validation, а НЕ на test (перевірка, що lgb_train сет валідації = val, не test):
   ✅ best_iteration=2, обрано за valid_sets=[lgb_val] (не test)

5. Test-скори НЕ використовувались до цієї комірки для жодного рішення (early stopping/threshold):
   ✅ test-прогнози згенеровано в Cell 11, ПІСЛЯ фіксації best_iteration у Cell 10

6. Розміри train/val/test збігаються з очікуваними зі Step 4A.2:
   train=249,725 (= 249,725)  val=15,192 (= 15,192)  test=65,159 (= 65,159)

✅ Усі leakage/sanity-перевірки пройдено.


In [15]:
# ---------- Cell 16: Збереження результатів ----------
OUT_DIR = Path("/srv/ids_research/data/cti/results")
OUT_DIR.mkdir(exist_ok=True)

summary_df.to_csv(OUT_DIR / f"step4a4_baseline_summary_h{HORIZON}.csv")
main_df.to_csv(OUT_DIR / f"step4a4_main_metrics_h{HORIZON}.csv", index=False)
topk_df.to_csv(OUT_DIR / f"step4a4_topk_metrics_h{HORIZON}.csv", index=False)
b2s_model.save_model(str(OUT_DIR / f"step4a4_b2s_model_h{HORIZON}.txt"))

print(f"Збережено в {OUT_DIR}:")
for f in sorted(OUT_DIR.glob(f"step4a4_*h{HORIZON}*")):
    print(" ", f.name)

Збережено в /srv/ids_research/data/cti/results:
  step4a4_b2s_model_h180.txt
  step4a4_baseline_summary_h180.csv
  step4a4_main_metrics_h180.csv
  step4a4_topk_metrics_h180.csv


## Step 4A.4 — Diagnostic A/B (ПІСЛЯ основного baseline, TEST не чіпається)

Мета: зрозуміти, чи `best_iteration=2` — реальна властивість B2_S, чи артефакт дуже малої
кількості позитивів на validation (n=27). Diagnostic A дивиться на розподіли ознак і
feature importance; Diagnostic B тренує B2_S з кількома фіксованими бюджетами ітерацій
(без early stopping) і оцінює **лише на validation**. Жодна з цих комірок НЕ звертається
до `test` — заморожений протокол одноразової фінальної оцінки лишається недоторканим.

In [16]:
# ================================================================
# Diagnostic A: distributions, feature importance, rank similarity
# ================================================================

print("=" * 70)
print("Diagnostic A.1 - розподіли B2_S ознак (train)")
print("=" * 70)

for col in B2_S_FEATURES:
    print(f"\n{col}:")
    print(train[col].describe())


print("\n" + "=" * 70)
print("Diagnostic A.2 - feature importance B2_S (gain-based)")
print("=" * 70)

importance = pd.Series(
    b2s_model.feature_importance(importance_type="gain"),
    index=B2_S_FEATURES
).sort_values(ascending=False)

importance_pct = 100 * importance / importance.sum()

print("Absolute gain:")
print(importance)

print("\nRelative gain (%):")
print(importance_pct)

print(
    "\n\u2139 Gain-based importance характеризує внесок ознак "
    "у зменшення objective через split-и. "
    "Низька importance не означає автоматично повну "
    "відсутність впливу ознаки на прогноз."
)


print("\n" + "=" * 70)
print("Diagnostic A.3 - Spearman rank correlation "
      "між B1 та B2_S (validation)")
print("=" * 70)

from scipy.stats import spearmanr

rho_val, p_val = spearmanr(
    b1_val_scores,
    b2s_val_scores
)

print(f"Validation Spearman rho = {rho_val:.4f}")
print(f"p-value = {p_val:.4g}")

print(
    "\n\u2139 Spearman rho характеризує схожість ranking "
    "між raw EPSS та B2_S prediction. "
    "Значення, близьке до 1, означає подібний порядок; "
    "нижче значення вказує на суттєву перебудову ranking. "
    "Ця діагностика сама по собі не визначає, "
    "яка саме ознака спричинила перебудову."
)


Diagnostic A.1 - розподіли B2_S ознак (train)

epss_t:
count    249725.000000
mean          0.028132
std           0.083967
min           0.000250
25%           0.004160
50%           0.008900
75%           0.013740
max           0.994570
Name: epss_t, dtype: float64

cve_age_days:
count    249725.000000
mean       1741.619297
std        2154.325317
min           0.000000
25%          21.000000
50%         776.000000
75%        2841.000000
max       12203.000000
Name: cve_age_days, dtype: float64

post_epss_published:
count    249725.000000
mean          0.354566
std           0.478383
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max           1.000000
Name: post_epss_published, dtype: float64

Diagnostic A.2 - feature importance B2_S (gain-based)
Absolute gain:
epss_t                 1.447532e+10
cve_age_days           4.033609e+08
post_epss_published    1.550501e+08
dtype: float64

Relative gain (%):
epss_t                 96.285612
cve_

In [17]:
# ================================================================
# Diagnostic B: fixed boosting budgets - VALIDATION ONLY
# ================================================================
# Мета: описати залежність validation performance від фіксованого числа
# boosting iterations та перевірити, чи спостерігається рання деградація
# ranking performance. Test не використовується.

budget_results = []

for n_est in [1, 2, 5, 10, 20, 50]:

    m = lgb.train(
        params,
        lgb_train,
        num_boost_round=n_est
    )

    val_pred = m.predict(X_val)

    ap_val = average_precision_score(
        y_val,
        val_pred
    )

    roc_val = (
        roc_auc_score(y_val, val_pred)
        if len(np.unique(y_val)) > 1
        else np.nan
    )

    budget_results.append({
        "n_estimators": n_est,
        "val_ap": ap_val,
        "val_roc_auc": roc_val
    })


budget_df = pd.DataFrame(budget_results)

print("=" * 70)
print("Diagnostic B - B2_S validation performance "
      "при фіксованих boosting budgets")
print("=" * 70)

display(budget_df)


b1_val_ap_ref = main_df[
    (main_df["model"] == "B1") &
    (main_df["split"] == "val")
]["ap"].iloc[0]

print("")
print(f"B1 (raw EPSS) validation AP = {b1_val_ap_ref:.4f}")

print(
    "\n\u2139 Це описова діагностика на validation. "
    "Test не використовується. "
    "Фіксовані boosting budgets не змінюють "
    "best_iteration основної B2_S моделі."
)


Diagnostic B - B2_S validation performance при фіксованих boosting budgets


,n_estimators,val_ap,val_roc_auc
0,1,0.053977,0.622707
1,2,0.055299,0.613966
2,5,0.001744,0.499664
3,10,0.001741,0.490702
4,20,0.001996,0.520692
5,50,0.002386,0.589453



B1 (raw EPSS) validation AP = 0.0756

ℹ Це описова діагностика на validation. Test не використовується. Фіксовані boosting budgets не змінюють best_iteration основної B2_S моделі.


## Diagnostic C — оцінка внеску окремих temporally safe ознак у validation ranking performance

Чотири комбінації ознак × {1, 2, 5} дерев, оцінка ЛИШЕ на validation. Test не чіпається —
результати B0/B1/B2_S для фінальної таблиці вже заморожені. Перша комбінація названа
`EPSS_only_LGBM` (не `B1`), щоб не плутати з основним B1 (сирий epss_t без навчання) —
тут це LightGBM, навчений лише на epss_t, що є ІНШОЮ моделлю.

In [18]:
# ---------- Diagnostic C: feature ablation (VALIDATION ONLY) ----------
# Мета: оцінити внесок окремих temporally safe ознак
# у validation ranking performance.
# Test не використовується.
#
# "EPSS_only_LGBM" - LightGBM, навчений лише на epss_t. Це ІНША модель, ніж основний
# B1 (сирий epss_t без будь-якого навчання) - назва навмисне відрізняється від "B1",
# щоб не створювати плутанини у статті.
#
# scale_pos_weight беремо з уже зафіксованого params (розрахований з train, однаковий
# для всіх комбінацій ознак - train/val склад не змінюється між абляціями).

ABLATION_FEATURE_SETS = {
    "EPSS_only_LGBM": ["epss_t"],
    "EPSS_age": ["epss_t", "cve_age_days"],
    "EPSS_cohort": ["epss_t", "post_epss_published"],
    "B2_S": ["epss_t", "cve_age_days", "post_epss_published"],
}

N_TREES_GRID = [1, 2, 5]

ablation_results = []

for combo_name, feats in ABLATION_FEATURE_SETS.items():

    X_train_c = train[feats]
    X_val_c = val[feats]

    lgb_train_c = lgb.Dataset(
        X_train_c,
        label=y_train
    )

    for n_est in N_TREES_GRID:

        m = lgb.train(
            params,
            lgb_train_c,
            num_boost_round=n_est
        )

        val_pred = m.predict(X_val_c)

        ap_val = average_precision_score(
            y_val,
            val_pred
        )

        roc_val = (
            roc_auc_score(y_val, val_pred)
            if len(np.unique(y_val)) > 1
            else np.nan
        )

        ablation_results.append({
            "combo": combo_name,
            "features": ", ".join(feats),
            "n_estimators": n_est,
            "val_ap": ap_val,
            "val_roc_auc": roc_val
        })


ablation_df = pd.DataFrame(ablation_results)

print("=" * 70)
print("Diagnostic C - feature ablation (VALIDATION ONLY)")
print("=" * 70)

print("\nValidation AP:")
display(
    ablation_df.pivot(
        index="combo",
        columns="n_estimators",
        values="val_ap"
    )
)

print("\nValidation ROC-AUC:")
display(
    ablation_df.pivot(
        index="combo",
        columns="n_estimators",
        values="val_roc_auc"
    )
)

print(
    "\n\u2139 Це описова ablation-діагностика на validation. "
    "Test не використовується і не змінює заморожені "
    "результати B0/B1/B2_S у фінальній таблиці."
)

print(
    "\n\u2139 Мета - оцінити внесок cve_age_days та "
    "post_epss_published окремо та разом з epss_t "
    "у validation ranking performance."
)


Diagnostic C - feature ablation (VALIDATION ONLY)

Validation AP:


n_estimators,1,2,5
combo,,,
B2_S,0.053977,0.055299,0.001744
EPSS_age,0.053382,0.053447,0.001853
EPSS_cohort,0.053781,0.001592,0.003044
EPSS_only_LGBM,0.036379,0.001413,0.001728



Validation ROC-AUC:


n_estimators,1,2,5
combo,,,
B2_S,0.622707,0.613966,0.499664
EPSS_age,0.532632,0.545332,0.479064
EPSS_cohort,0.612633,0.428542,0.605853
EPSS_only_LGBM,0.526388,0.342776,0.461286



ℹ Це описова ablation-діагностика на validation. Test не використовується і не змінює заморожені результати B0/B1/B2_S у фінальній таблиці.

ℹ Мета - оцінити внесок cve_age_days та post_epss_published окремо та разом з epss_t у validation ranking performance.


In [20]:
# ---------- Зведений звіт: усі результати Step 4A.4 в одному місці ----------
import sys

REPORT_SECTIONS = {
    "main_metrics (B0/B1/B2_S, val+test)": "main_df",
    "topk_metrics (Precision/Recall@K, val+test)": "topk_df",
    "summary_test (фінальна таблиця для статті)": "summary_df",
    "diagnostic_B_fixed_budgets (val only)": "budget_df",
    "diagnostic_C_feature_ablation (val only)": "ablation_df",
}

collected = {}
missing = []
for label, varname in REPORT_SECTIONS.items():
    obj = globals().get(varname)
    if obj is None:
        missing.append(label)
        continue
    collected[label] = obj

print("=" * 70)
print("ЗВЕДЕНИЙ ЗВІТ — Step 4A.4 (H={})".format(HORIZON))
print("=" * 70)

for label, df in collected.items():
    print(f"\n--- {label} ---")
    display(df)

if missing:
    print("\n⚠ Пропущено (комірка ще не виконувалась у цій сесії):")
    for m in missing:
        print(f"  - {m}")

# Diagnostic A (importance/Spearman) зберігаються як окремі об'єкти, не DataFrame — додаємо окремо
diag_a_present = "importance" in globals() and "rho_val" in globals()
if diag_a_present:
    print("\n--- diagnostic_A_feature_importance (gain, %) ---")
    display(pd.DataFrame({"gain": importance, "gain_pct": importance_pct}))
    print(f"\n--- diagnostic_A_spearman_rho (B1 vs B2_S, validation) ---")
    print(f"rho={rho_val:.4f}, p={p_val:.4g}")
else:
    print("\n⚠ Пропущено: diagnostic_A_feature_importance / spearman (Diagnostic A ще не виконувався)")

ЗВЕДЕНИЙ ЗВІТ — Step 4A.4 (H=180)

--- main_metrics (B0/B1/B2_S, val+test) ---


,model,split,ap,roc_auc
0,B0,val,0.001777,0.500000
1,B0,test,0.001381,0.500000
2,B1,val,0.075574,0.539244
3,B1,test,0.018597,0.715477
4,B2_S,val,0.055299,0.613966
5,B2_S,test,0.007880,0.573759



--- topk_metrics (Precision/Recall@K, val+test) ---


,model,split,top_pct,K,precision,recall
0,B0,val,0.1%,15,0.001777,0.000987
1,B0,val,0.5%,76,0.001777,0.005003
2,B0,val,1.0%,152,0.001777,0.010005
3,B0,val,5.0%,760,0.001777,0.050026
4,B0,test,0.1%,65,0.001381,0.000998
5,B0,test,0.5%,326,0.001381,0.005003
6,B0,test,1.0%,652,0.001381,0.010006
7,B0,test,5.0%,3258,0.001381,0.050001
8,B1,val,0.1%,15,0.200000,0.111111
9,B1,val,0.5%,76,0.065789,0.185185



--- summary_test (фінальна таблиця для статті) ---


...


--- diagnostic_C_feature_ablation (val only) ---


,combo,features,n_estimators,val_ap,val_roc_auc
0,EPSS_only_LGBM,epss_t,1,0.036379,0.526388
1,EPSS_only_LGBM,epss_t,2,0.001413,0.342776
2,EPSS_only_LGBM,epss_t,5,0.001728,0.461286
3,EPSS_age,"epss_t, cve_age_days",1,0.053382,0.532632
4,EPSS_age,"epss_t, cve_age_days",2,0.053447,0.545332
5,EPSS_age,"epss_t, cve_age_days",5,0.001853,0.479064
6,EPSS_cohort,"epss_t, post_epss_published",1,0.053781,0.612633
7,EPSS_cohort,"epss_t, post_epss_published",2,0.001592,0.428542
8,EPSS_cohort,"epss_t, post_epss_published",5,0.003044,0.605853
9,B2_S,"epss_t, cve_age_days, post_epss_published",1,0.053977,0.622707



--- diagnostic_A_feature_importance (gain, %) ---


,gain,gain_pct
epss_t,1.447532e+10,96.285612
cve_age_days,4.033609e+08,2.683040
post_epss_published,1.550501e+08,1.031348



--- diagnostic_A_spearman_rho (B1 vs B2_S, validation) ---
rho=0.2321, p=7.213e-185


In [22]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openpyxl", "tabulate"], check=True)

CompletedProcess(args=['/home/ivan_r_opirskyy/ids_research/bin/python', '-m', 'pip', 'install', '-q', 'openpyxl', 'tabulate'], returncode=0)

In [24]:
import os, sys, subprocess
# ---------- Збереження зведеного звіту: один Excel (кілька листів) + один Markdown ----------
try:
    import openpyxl  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openpyxl"], check=True)
try:
    import tabulate  # noqa: F401 — потрібен для df.to_markdown()
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tabulate"], check=True)

REPORT_OUT_DIR = Path(os.environ.get(
    "STEP4A4_OUT_DIR", str(Path.home() / "notebooks" / "2 article" / "release" / "results")))
REPORT_OUT_DIR.mkdir(parents=True, exist_ok=True)

xlsx_path = REPORT_OUT_DIR / f"step4a4_full_report_h{HORIZON}.xlsx"
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    for label, df in collected.items():
        sheet_name = label.split(" (")[0][:31]   # обмеження Excel: назва листа <= 31 символ
        df.to_excel(writer, sheet_name=sheet_name)
    if diag_a_present:
        pd.DataFrame({"gain": importance, "gain_pct": importance_pct}).to_excel(
            writer, sheet_name="diagnostic_A_importance")
        pd.DataFrame([{"rho": rho_val, "p_value": p_val}]).to_excel(
            writer, sheet_name="diagnostic_A_spearman", index=False)

print(f"Excel-звіт збережено: {xlsx_path}")

md_lines = [f"# Step 4A.4 — Full Report (H={HORIZON})\n"]
md_lines.append("Primary temporally-safe baseline comparison: B0 / B1 / B2_S. "
                 "Diagnostics A/B/C are validation-only, exploratory (test untouched).\n")

if "summary_df" in collected:
    md_lines.append("## Final test summary (frozen, one-time evaluation)\n")
    md_lines.append(collected["summary_df"].to_markdown() + "\n")
if "main_metrics (B0/B1/B2_S, val+test)" in collected:
    md_lines.append("## Main metrics (val + test)\n")
    md_lines.append(collected["main_metrics (B0/B1/B2_S, val+test)"].to_markdown(index=False) + "\n")
if diag_a_present:
    md_lines.append("## Diagnostic A — feature importance (gain, %) and rank correlation\n")
    md_lines.append(pd.DataFrame({"gain": importance, "gain_pct": importance_pct}).to_markdown() + "\n")
    md_lines.append(f"\nSpearman rho (B1 vs B2_S, validation) = {rho_val:.4f} (p={p_val:.4g})\n")
if "diagnostic_B_fixed_budgets (val only)" in collected:
    md_lines.append("## Diagnostic B — fixed boosting budgets (validation only)\n")
    md_lines.append(collected["diagnostic_B_fixed_budgets (val only)"].to_markdown(index=False) + "\n")
if "diagnostic_C_feature_ablation (val only)" in collected:
    md_lines.append("## Diagnostic C — feature ablation (validation only)\n")
    md_lines.append(collected["diagnostic_C_feature_ablation (val only)"].to_markdown() + "\n")

md_path = REPORT_OUT_DIR / f"step4a4_full_report_h{HORIZON}.md"
md_path.write_text("\n".join(md_lines), encoding="utf-8")
print(f"Markdown-звіт збережено: {md_path}")

Excel-звіт збережено: /home/ivan_r_opirskyy/notebooks/2 article/release/results/step4a4_full_report_h180.xlsx
Markdown-звіт збережено: /home/ivan_r_opirskyy/notebooks/2 article/release/results/step4a4_full_report_h180.md
